In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

oot_scores = pd.read_parquet('../data/processed/oot_scores.parquet')

In [2]:
df=oot_scores.copy()
score_col='score'
target_col='target'

total_good = (df[target_col] == 0).sum()
total_bad = (df[target_col] == 1).sum()

print(f"总样本数: {len(df)}")
print(f"好客户数: {total_good}")
print(f"坏客户数: {total_bad}")
print(f"整体坏账率: {total_bad / len(df):.2%}")

总样本数: 61503
好客户数: 56538
坏客户数: 4965
整体坏账率: 8.07%


In [3]:
min_score = df[score_col].min()
max_score = df[score_col].max()
thresholds = np.arange(min_score, max_score + 1, 5)

In [4]:
results = []
for cutoff in thresholds:
    # 分数 >= cutoff 的客户被批准
    approved = df[df[score_col] >= cutoff]
    rejected = df[df[score_col] < cutoff]
    
    # 通过率
    approval_rate = len(approved) / len(df)
    
    # 坏账率（批准的客户里，坏客户占比）
    if len(approved) > 0:
        bad_rate = approved[target_col].mean()
    else:
        bad_rate = 0
    
    # 误杀率（被拒绝的好客户 / 总好客户）
    false_positive = ((rejected[target_col] == 0).sum()) / total_good
    
    # 漏杀率（被批准的坏客户 / 总坏客户）
    false_negative = ((approved[target_col] == 1).sum()) / total_bad
    
    results.append({
        'cutoff': cutoff,
        'approval_rate': approval_rate,
        'bad_rate': bad_rate,
        'false_positive_rate': false_positive,
        'false_negative_rate': false_negative
    })

result_df = pd.DataFrame(results)
print(result_df.head(10))

   cutoff  approval_rate  bad_rate  false_positive_rate  false_negative_rate
0     417       1.000000  0.080728             0.000000             1.000000
1     422       0.999967  0.080698             0.000000             0.999597
2     427       0.999756  0.080552             0.000053             0.997583
3     432       0.999463  0.080429             0.000212             0.995770
4     437       0.998878  0.080086             0.000424             0.990937
5     442       0.997951  0.079639             0.000867             0.984491
6     447       0.996212  0.078962             0.001875             0.974421
7     452       0.993155  0.077781             0.003661             0.956898
8     457       0.989431  0.076726             0.006261             0.940383
9     462       0.984228  0.075199             0.009852             0.916818


In [5]:
# 1. 去掉通过率为 0 的行（这些点没有业务意义，且会导致坏账率计算异常）
results_df = result_df[result_df['approval_rate'] > 0.05].copy()



# 2. 按 Cutoff 升序排列（这是防止线条乱飞的核心）
results_df = results_df.sort_values(by='cutoff').reset_index(drop=True)

print(f"清洗前: {len(result_df)} 行")
print(f"清洗后: {len(results_df)} 行")
print(results_df.head(10))
print("...")
print(results_df.tail(5))
# ==========================================

清洗前: 46 行
清洗后: 32 行
   cutoff  approval_rate  bad_rate  false_positive_rate  false_negative_rate
0     417       1.000000  0.080728             0.000000             1.000000
1     422       0.999967  0.080698             0.000000             0.999597
2     427       0.999756  0.080552             0.000053             0.997583
3     432       0.999463  0.080429             0.000212             0.995770
4     437       0.998878  0.080086             0.000424             0.990937
5     442       0.997951  0.079639             0.000867             0.984491
6     447       0.996212  0.078962             0.001875             0.974421
7     452       0.993155  0.077781             0.003661             0.956898
8     457       0.989431  0.076726             0.006261             0.940383
9     462       0.984228  0.075199             0.009852             0.916818
...
    cutoff  approval_rate  bad_rate  false_positive_rate  false_negative_rate
27     552       0.208201  0.016634             0.7

In [6]:
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.plot(results_df['cutoff'], results_df['approval_rate'] * 100, 
         'b-', linewidth=2, label='通过率 (Approval Rate)')
ax1.set_xlabel('Cutoff 分数阈值')
ax1.set_ylabel('通过率 (%)', color='b')
ax1.set_ylim(bottom=0)
ax1.tick_params(axis='y', labelcolor='b')

ax2 = ax1.twinx()
ax2.plot(results_df['cutoff'], results_df['bad_rate'] * 100, 
         'r-', linewidth=2, label='坏账率 (Bad Rate)')
ax2.set_ylabel('坏账率 (%)', color='r')
ax2.set_ylim(bottom=0)
ax2.tick_params(axis='y', labelcolor='r')

plt.title('通过率-坏账率权衡曲线 (Trade-off Curve)')
fig.legend(loc='upper right', bbox_to_anchor=(0.85, 0.85))

example_cutoffs = [480, 530, 580]
example_labels = ['激进', '均衡', '保守']
for cutoff, label in zip(example_cutoffs, example_labels):
    closest_idx = results_df['cutoff'].sub(cutoff).abs().idxmin()
    row = results_df.loc[closest_idx]
    ar = row['approval_rate'] * 100
    br = row['bad_rate'] * 100
    ax1.scatter(cutoff, ar, c='b', s=50, zorder=5)
    ax2.scatter(cutoff, br, c='r', s=50, zorder=5)
    ax1.annotate(f'{label}\n(AR:{ar:.1f}%, BR:{br:.1f}%)', 
                 (cutoff, ar), textcoords="offset points", xytext=(0,10), ha='center')

plt.grid(True, alpha=0.3)
plt.show()

C:\Users\86186\AppData\Local\Temp\claude-agent-tmp\ipykernel_23780\245041510.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
target_rates = [0.75,0.60,0.45]#目标通过率：75%，60%，45%
strategies = []
for target_ar in target_rates:
    closest_idx = (results_df['approval_rate'] - target_ar).abs().idxmin()
    row = results_df.loc[closest_idx]
    strategies.append({
        '策略': f'通过率≈{target_ar:.0%}',
        'Cutoff': int(row['cutoff']),
        '通过率': f"{row['approval_rate']:.1%}",
        '坏账率': f"{row['bad_rate']:.1%}",
        '误杀率': f"{row['false_positive_rate']:.1%}",
        '漏杀率': f"{row['false_negative_rate']:.1%}"
    })

strategies_df = pd.DataFrame(strategies)
print("\n===== 三档策略对比 =====")
print(strategies_df)


===== 三档策略对比 =====
        策略  Cutoff    通过率   坏账率    误杀率    漏杀率
0  通过率≈75%     507  76.1%  4.5%  20.9%  42.2%
1  通过率≈60%     522  58.7%  3.2%  38.2%  23.3%
2  通过率≈45%     532  45.5%  2.5%  51.7%  14.0%


In [8]:
import matplotlib.pyplot as plt

# ===== 将上一单元计算出的三档直接通过率候选点用于可视化 =====
# 这些点只展示阈值变化趋势，最终推荐由后面的业务约束优化决定。
three_strategies = pd.DataFrame({
    '策略': ['激进候选', '中间候选', '保守候选'],
    'Cutoff': strategies_df['Cutoff'].to_numpy(),
    'AR': strategies_df['通过率'].str.rstrip('%').astype(float).to_numpy(),
    'BR': strategies_df['坏账率'].str.rstrip('%').astype(float).to_numpy(),
})

fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.plot(results_df['cutoff'], results_df['approval_rate'] * 100,
         'b-', linewidth=2.5, label='通过率 (Approval Rate)')
ax1.set_xlabel('Cutoff 分数阈值', fontsize=12)
ax1.set_ylabel('通过率 (%)', color='b', fontsize=12)
ax1.tick_params(axis='y', labelcolor='b')
ax1.set_ylim(bottom=0, top=105)

ax2 = ax1.twinx()
ax2.plot(results_df['cutoff'], results_df['bad_rate'] * 100,
         'r-', linewidth=2.5, label='坏账率 (Bad Rate)')
ax2.set_ylabel('坏账率 (%)', color='r', fontsize=12)
ax2.tick_params(axis='y', labelcolor='r')
ax2.set_ylim(bottom=0, top=9)

colors_map = {'激进候选': '#FF6B35', '中间候选': '#004E89', '保守候选': '#1A936F'}
for _, row in three_strategies.iterrows():
    c = row['Cutoff']
    ar = row['AR']
    br = row['BR']
    color = colors_map[row['策略']]

    ax1.scatter(c, ar, c=color, s=120, zorder=5, edgecolors='white', linewidths=1.5)
    ax2.scatter(c, br, c=color, s=120, zorder=5, edgecolors='white', linewidths=1.5)

    ax1.annotate(f"{row['策略']}\nAR={ar:.1f}%", (c, ar),
                 textcoords="offset points", xytext=(0, 12),
                 ha='center', fontsize=10, fontweight='bold', color=color)
    ax2.annotate(f"BR={br:.1f}%", (c, br),
                 textcoords="offset points", xytext=(0, -15),
                 ha='center', fontsize=10, fontweight='bold', color=color)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2,
           loc='upper right', fontsize=10, framealpha=0.9)

plt.title('通过率-坏账率权衡曲线 — 三档候选点（非最终推荐）', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig('../models/tradeoff_3strategies.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\86186\AppData\Local\Temp\claude-agent-tmp\ipykernel_23780\288631201.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# ===== 全景扫描：每10%取一个点 =====
target_rates1 = [0.90, 0.80, 0.70, 0.60, 0.50, 0.40]  #目标通过率：90%，80%，70%，60%，50%，40%
strategies = []
for target_ar in target_rates1:
    closest_idx = (results_df['approval_rate'] - target_ar).abs().idxmin()
    row = results_df.loc[closest_idx]
    strategies.append({
        '策略档位': f'AR={target_ar:.0%}',
        'Cutoff'  : int(row['cutoff']),
        '通过率'  : f"{row['approval_rate']:.1%}",
        '坏账率'  : f"{row['bad_rate']:.1%}",
        '误杀率'  : f"{row['false_positive_rate']:.1%}",  # 好客户被错杀的比例
        '漏杀率'  : f"{row['false_negative_rate']:.1%}"   # 坏客户被放过的风险
    })
    
strategies_full_df = pd.DataFrame(strategies)
print("====== 多档位策略挖掘结果 ======")
print(strategies_full_df)

====== 多档位策略挖掘结果 ======
     策略档位  Cutoff    通过率   坏账率    误杀率    漏杀率
0  AR=90%     487  91.0%  6.1%   7.0%  68.7%
1  AR=80%     502  80.8%  4.9%  16.4%  48.9%
2  AR=70%     512  70.7%  4.0%  26.1%  34.8%
3  AR=60%     522  58.7%  3.2%  38.2%  23.3%
4  AR=50%     527  52.1%  2.8%  44.9%  18.4%
5  AR=40%     537  38.8%  2.3%  58.8%  11.0%


In [10]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.plot(results_df['cutoff'], results_df['approval_rate'] * 100, 
         'b-', linewidth=2, label='通过率 (Approval Rate)')
ax1.set_xlabel('Cutoff 分数阈值')
ax1.set_ylabel('通过率 (%)', color='b')
ax1.tick_params(axis='y', labelcolor='b')
ax1.set_ylim(bottom=0, top=105)

ax2 = ax1.twinx()
ax2.plot(results_df['cutoff'], results_df['bad_rate'] * 100, 
         'r-', linewidth=2, label='坏账率 (Bad Rate)')
ax2.set_ylabel('坏账率 (%)', color='r')
ax2.tick_params(axis='y', labelcolor='r')
ax2.set_ylim(bottom=0)

for _, row in strategies_full_df.iterrows():
    cutoff_val = row['Cutoff']
    ar = float(row['通过率'].strip('%'))
    br = float(row['坏账率'].strip('%'))
    
    ax1.scatter(cutoff_val, ar, c='b', s=60, zorder=5)
    ax2.scatter(cutoff_val, br, c='r', s=60, zorder=5)
    
    ax1.annotate(f"AR={ar:.0f}%", (cutoff_val, ar),
                 textcoords="offset points", xytext=(0, 10), ha='center', fontsize=8, color='blue')
    ax2.annotate(f"BR={br:.1f}%", (cutoff_val, br),
                 textcoords="offset points", xytext=(0, -12), ha='center', fontsize=8, color='red')

colors = ['#FF6B35', '#004E89', '#1A936F']  # 橙、蓝、绿
ax1.fill_between(results_df['cutoff'], results_df['approval_rate'] * 100,
alpha=0.1, color='blue')
ax2.fill_between(results_df['cutoff'], results_df['bad_rate'] * 100,
alpha=0.1, color='red')

plt.title('通过率-坏账率权衡曲线 (Trade-off Curve) - 全景策略标注', fontsize=14)
fig.legend(loc='upper right', bbox_to_anchor=(0.88, 0.88))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

C:\Users\86186\AppData\Local\Temp\claude-agent-tmp\ipykernel_23780\1063428053.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# ===== 业务约束下自动选择最优 Cutoff =====
MAX_BAD_RATE = 0.03       # 批准客户坏账率不超过 3%
MIN_APPROVAL_RATE = 0.35  # 整体通过率不低于 35%
HIGH_THRESHOLD = 560      # 高分快速通过线
MANUAL_PASS_RATE = 0.50   # 人工复核预计通过率
AVG_LOAN = 10000          # 单笔平均放款额（元）
PROFIT_RATE = 0.07        # 固定利率场景单笔利润率
BAD_LOSS_RATE = 0.85      # 坏账损失率

auto_pass_for_optimization = df[df['score'] >= HIGH_THRESHOLD]
cutoff_candidates = []

# 枚举每一个整数 Cutoff，先施加风险与规模约束，再以净利润最大化选优
for candidate_cutoff in range(int(df['score'].min()), HIGH_THRESHOLD + 1):
    manual_for_optimization = df[(df['score'] >= candidate_cutoff) &
                                 (df['score'] < HIGH_THRESHOLD)]
    approved_count = (len(auto_pass_for_optimization) +
                      len(manual_for_optimization) * MANUAL_PASS_RATE)
    bad_count = (auto_pass_for_optimization['target'].sum() +
                 manual_for_optimization['target'].sum() * MANUAL_PASS_RATE)
    approval_rate = approved_count / len(df)
    bad_rate = bad_count / approved_count
    loan_volume = approved_count * AVG_LOAN
    revenue = loan_volume * PROFIT_RATE
    bad_loss = bad_count * AVG_LOAN * BAD_LOSS_RATE
    net_profit = revenue - bad_loss
    roi = net_profit / loan_volume

    cutoff_candidates.append({
        'cutoff': candidate_cutoff,
        'approval_rate': approval_rate,
        'bad_rate': bad_rate,
        'approved_count': approved_count,
        'bad_count': bad_count,
        'net_profit': net_profit,
        'roi': roi,
    })

cutoff_optimization_df = pd.DataFrame(cutoff_candidates)
feasible_cutoffs = cutoff_optimization_df[
    (cutoff_optimization_df['bad_rate'] <= MAX_BAD_RATE) &
    (cutoff_optimization_df['approval_rate'] >= MIN_APPROVAL_RATE)
].copy()

if feasible_cutoffs.empty:
    raise ValueError('没有 Cutoff 同时满足当前坏账率和通过率约束')

feasible_cutoffs = feasible_cutoffs.sort_values(
    ['net_profit', 'roi'], ascending=False
).reset_index(drop=True)
optimal_strategy = feasible_cutoffs.iloc[0]
optimal_cutoff = int(optimal_strategy['cutoff'])

print('===== Cutoff 约束优化结果 =====')
print(f'约束：坏账率 <= {MAX_BAD_RATE:.0%}，整体通过率 >= {MIN_APPROVAL_RATE:.0%}')
print('优化目标：满足约束后的净利润最大化')
print(f'最优 Cutoff：{optimal_cutoff}')
print(f"整体通过率：{optimal_strategy['approval_rate']:.2%}")
print(f"批准客户坏账率：{optimal_strategy['bad_rate']:.2%}")
print(f"预计净利润：{optimal_strategy['net_profit'] / 10000:.2f} 万元")
print(f"预计 ROI：{optimal_strategy['roi']:.2%}")
display(feasible_cutoffs.head(10))

# ===== 使用自动选出的 Cutoff 进行三段式决策区间统计 =====
cutoff = optimal_cutoff
high_threshold = HIGH_THRESHOLD
auto_pass = df[df['score'] >= high_threshold]
manual_review = df[(df['score'] >= cutoff) & (df['score'] < high_threshold)]
auto_reject = df[df['score'] < cutoff]
segments = {
    f'自动通过 (score>={high_threshold})': auto_pass,
    f'人工复核 ({cutoff}<=score<{high_threshold})': manual_review,
    f'自动拒绝 (score<{cutoff})': auto_reject,
}

print(f'\n===== 三段式决策区间统计（约束优化策略 Cutoff={cutoff}）=====\n')
for name, seg in segments.items():
    total = len(seg)
    bad_count = seg['target'].sum()
    good_count = total - bad_count
    bad_rate = bad_count / total * 100 if total > 0 else 0
    print(f'{name}:')
    print(f'  总人数: {total}')
    print(f'  好客户数: {good_count}')
    print(f'  坏客户数: {bad_count}')
    print(f'  坏账率: {bad_rate:.2f}%')
    print()

approved = len(auto_pass) + len(manual_review) * MANUAL_PASS_RATE
expected_bad = (auto_pass['target'].sum() +
                manual_review['target'].sum() * MANUAL_PASS_RATE)
print(f'整体通过率（人工复核预计通过 {MANUAL_PASS_RATE:.0%}）：{approved / len(df):.2%}')
print(f'批准客户坏账率：{expected_bad / approved:.2%}')

===== Cutoff 约束优化结果 =====
约束：坏账率 <= 3%，整体通过率 >= 35%
优化目标：满足约束后的净利润最大化
最优 Cutoff：520
整体通过率：37.33%
批准客户坏账率：2.97%
预计净利润：1028.49 万元
预计 ROI：4.48%


,cutoff,approval_rate,bad_rate,approved_count,bad_count,net_profit,roi
0,520,0.373348,0.029658,22962.0,681.0,10284900.0,0.044791
1,521,0.366941,0.028957,22568.0,653.5,10242850.0,0.045387
2,522,0.360852,0.028725,22193.5,637.5,10116700.0,0.045584
3,523,0.354381,0.028102,21795.5,612.5,10050600.0,0.046113



===== 三段式决策区间统计（约束优化策略 Cutoff=520）=====

自动通过 (score>=560):
  总人数: 8266
  好客户数: 8149
  坏客户数: 117
  坏账率: 1.42%

人工复核 (520<=score<560):
  总人数: 29392
  好客户数: 28264
  坏客户数: 1128
  坏账率: 3.84%

自动拒绝 (score<520):
  总人数: 23845
  好客户数: 20125
  坏客户数: 3720
  坏账率: 15.60%

整体通过率（人工复核预计通过 50%）：37.33%
批准客户坏账率：2.97%


In [12]:
# ===== 固定利率场景：三档策略效果测算 =====
# 约束优化策略的 Cutoff 来自上一单元，不再手工指定
strategies = {
    '激进': {'cutoff': 512, 'high': 550},
    '约束优化（推荐）': {'cutoff': optimal_cutoff, 'high': HIGH_THRESHOLD},
    '保守': {'cutoff': 537, 'high': 570},
}

results = []

for name, params in strategies.items():
    cutoff = params['cutoff']
    high = params['high']
    
    # 自动通过：score >= high
    auto_pass = df[df['score'] >= high]
    # 人工复核：cutoff <= score < high
    manual = df[(df['score'] >= cutoff) & (df['score'] < high)]
    # 人工复核按期望通过率计算，避免随机抽样带来的口径波动
    total_approved = len(auto_pass) + len(manual) * MANUAL_PASS_RATE
    total_pool = len(df)
    
    # 实际坏账人数（在通过的人中）
    bad_in_auto = auto_pass['target'].sum()
    bad_in_manual = manual['target'].sum() * MANUAL_PASS_RATE
    total_bad = bad_in_auto + bad_in_manual
    
    # 坏账率
    actual_br = total_bad / total_approved if total_approved > 0 else 0
    
    # 放款量
    loan_volume = total_approved * AVG_LOAN
    
    # 预计坏账损失
    bad_loss = total_bad * AVG_LOAN * BAD_LOSS_RATE
    
    # 预估收益
    revenue = total_approved * AVG_LOAN * PROFIT_RATE
    net_profit = revenue - bad_loss
    
    # 综合 ROI
    roi = (revenue - bad_loss) / loan_volume if loan_volume > 0 else 0
    
    results.append({
        '策略': name,
        'Cutoff': cutoff,
        '高分线': high,
        '放款人数': int(round(total_approved)),
        '通过率': f"{total_approved/total_pool:.1%}",
        '坏账人数': int(round(total_bad)),
        '实际坏账率': f"{actual_br:.1%}",
        '放款量(万元)': f"{loan_volume/10000:.1f}",
        '坏账损失(万元)': f"{bad_loss/10000:.1f}",
        '总收益(万元)': f"{revenue/10000:.1f}",
        '净利润(万元)': f"{net_profit/10000:.1f}",
        '综合ROI': f"{roi:.1%}",
    })

results_df_final = pd.DataFrame(results)
print("===== 三档策略效果测算 =====")
print(results_df_final.to_string(index=False))

===== 三档策略效果测算 =====
      策略  Cutoff  高分线  放款人数   通过率  坏账人数 实际坏账率 放款量(万元) 坏账损失(万元) 总收益(万元) 净利润(万元) 综合ROI
      激进     512  550 28854 46.9%   993  3.4% 28853.5    844.0  2019.7  1175.7  4.1%
约束优化（推荐）     520  560 22962 37.3%   681  3.0% 22962.0    578.9  1607.3  1028.5  4.5%
      保守     537  570 14052 22.8%   296  2.1% 14051.5    252.0   983.6   731.6  5.2%


In [13]:
# ===== 差异化定价场景 =====
funding_cost = 0.04  # 资金成本率 4%

def get_interest_rate(score):
    """使用约束优化策略确定的分层边界返回年化利率。"""
    if score >= HIGH_THRESHOLD:
        return 0.10
    elif score >= optimal_cutoff:
        return 0.15
    else:
        return 0.22

# 沿用固定利率场景的同一组准入策略
results_diff = []

for name, params in strategies.items():
    cutoff = params['cutoff']
    high = params['high']
    
    # 用 approval_weight 表示期望批准比例：自动通过=1，人工复核=50%
    auto_pass = df[df['score'] >= high].copy()
    auto_pass['approval_weight'] = 1.0
    manual = df[(df['score'] >= cutoff) & (df['score'] < high)].copy()
    manual['approval_weight'] = MANUAL_PASS_RATE
    approval_pool = pd.concat([auto_pass, manual], ignore_index=True)
    
    total_approved = approval_pool['approval_weight'].sum()
    total_pool = len(df)
    
    # 所有金额均乘 approval_weight，与固定利率场景保持完全相同的批准人群口径
    approval_pool['interest_rate'] = approval_pool['score'].apply(get_interest_rate)
    approval_pool['income'] = (AVG_LOAN * approval_pool['interest_rate'] *
                               approval_pool['approval_weight'])
    approval_pool['cost'] = AVG_LOAN * funding_cost * approval_pool['approval_weight']
    approval_pool['loss'] = (approval_pool['target'] * AVG_LOAN * BAD_LOSS_RATE *
                             approval_pool['approval_weight'])
    approval_pool['net_profit'] = (approval_pool['income'] -
                                   approval_pool['cost'] - approval_pool['loss'])
    
    total_income = approval_pool['income'].sum()
    total_cost = approval_pool['cost'].sum()
    total_loss = approval_pool['loss'].sum()
    total_net_profit = approval_pool['net_profit'].sum()
    
    bad_count = (approval_pool['target'] * approval_pool['approval_weight']).sum()
    actual_br = bad_count / total_approved if total_approved > 0 else 0
    
    # 放款量
    loan_volume = total_approved * AVG_LOAN
    
    # ROI = 净利润 / 放款量
    roi = total_net_profit / loan_volume if loan_volume > 0 else 0
    
    results_diff.append({
        '策略': name,
        'Cutoff': cutoff,
        '高分线': high,
        '放款人数': int(round(total_approved)),
        '通过率': f"{total_approved/total_pool:.1%}",
        '坏账人数': int(round(bad_count)),
        '实际坏账率': f"{actual_br:.1%}",
        '放款量(万元)': f"{loan_volume/10000:.1f}",
        '总收入(万元)': f"{total_income/10000:.1f}",
        '坏账损失(万元)': f"{total_loss/10000:.1f}",
        '净利润(万元)': f"{total_net_profit/10000:.1f}",
        '综合ROI': f"{roi:.1%}",
    })

results_diff_df = pd.DataFrame(results_diff)
print("===== 差异化定价后三档策略效果对比 =====")
print(results_diff_df.to_string(index=False))

===== 差异化定价后三档策略效果对比 =====
      策略  Cutoff  高分线  放款人数   通过率  坏账人数 实际坏账率 放款量(万元) 总收入(万元) 坏账损失(万元) 净利润(万元) 综合ROI
      激进     512  550 28854 46.9%   993  3.4% 28853.5  4119.4    844.0  2121.2  7.4%
约束优化（推荐）     520  560 22962 37.3%   681  3.0% 22962.0  3031.0    578.9  1533.7  6.7%
      保守     537  570 14052 22.8%   296  2.1% 14051.5  1794.4    252.0   980.3  7.0%


In [14]:
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 直接从上一步差异化定价结果表读取绘图数据，避免手工硬编码导致图表与计算结果不一致
strategies = results_diff_df['策略'].tolist()
net_profit = results_diff_df['净利润(万元)'].astype(float).to_numpy()
roi = results_diff_df['综合ROI'].str.rstrip('%').astype(float).to_numpy()
bad_rate = results_diff_df['实际坏账率'].str.rstrip('%').astype(float).to_numpy()

x = np.arange(len(strategies))
width = 0.25

fig, ax1 = plt.subplots(figsize=(10, 6))

color1 = '#5DADE2'  # 蓝色
rects1 = ax1.bar(x - width, net_profit, width, label='净利润 (万元)', color=color1, alpha=0.8)
ax1.set_ylabel('净利润 (万元)', fontsize=12, color=color1)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_ylim(0, 4000)

ax2 = ax1.twinx()
color2 = '#28B463'  # 绿色
line1 = ax2.plot(x, roi, 'o-', color=color2, linewidth=2, markersize=8, label='综合 ROI (%)')
ax2.set_ylabel('综合 ROI (%)', fontsize=12, color=color2)
ax2.tick_params(axis='y', labelcolor=color2)
ax2.set_ylim(0, 20)

ax3 = ax1.twinx()
color3 = '#E74C3C'  # 红色
line2 = ax3.plot(x, bad_rate, 's--', color=color3, linewidth=2, markersize=8, label='实际坏账率 (%)')
ax3.set_ylabel('实际坏账率 (%)', fontsize=12, color=color3)
ax3.tick_params(axis='y', labelcolor=color3)
ax3.spines['right'].set_position(('outward', 60)) 

ax1.set_xlabel('审批策略', fontsize=12)
ax1.set_xticks(x)
ax1.set_xticklabels(strategies)

lines = [rects1] + line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left')

plt.title('差异化定价策略效果多维对比', fontsize=14)
plt.tight_layout()
plt.savefig('../models/pricing_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

C:\Users\86186\AppData\Local\Temp\claude-agent-tmp\ipykernel_23780\3089996373.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
